In [2]:
# Install dependencies
%pip install anthropic python-dotenv





Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [10]:
# Define utlity functions for building messages and making requests
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})
    return messages


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})
    return messages


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)  # spread params into function arguments
    return message.content[0].text

In [14]:
# Make a start list of messages
messages = []

# Add in the intitial user question
messages = add_user_message(messages, "What is the capital of France?")

# Pass the list of messages into 'chat' method to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message to the list of messages
messages = add_assistant_message(messages, answer)

# Add in the user's follow-up question
add_user_message(messages, "What is the population of that city?")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)

answer

'The population of Paris varies depending on how you define the area:\n\n- **City of Paris proper**: Approximately 2.1 million people\n- **Greater Paris (Île-de-France region)**: Approximately 12.3 million people\n\nThe city proper refers to the administrative boundaries of Paris itself, while Greater Paris includes the surrounding metropolitan area. These figures are based on recent estimates, though exact numbers can vary slightly depending on the source and year of the census data.'

In [ ]:
messages = []

while True:
    user_input = input("> ")
    if user_input.lower() in ["exit", "quit"]:
        break
    print(">", user_input)

    add_user_message(messages, user_input)

    answer = chat(messages)

    add_assistant_message(messages, answer)

    print("---")
    print(answer)
    print("---")

> what is 1+1
---
1 + 1 = 2
---
> is it a prime number
---
Yes, 2 is a prime number. In fact, it's the smallest prime number and the only even prime number. A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself, and 2 fits this definition perfectly.
---
> whats the next one
---
The next prime number after 2 is 3.

3 is prime because it's only divisible by 1 and itself (3). After 3, the sequence of prime numbers continues: 5, 7, 11, 13, 17, 19, 23, 29, 31, and so on.
---
> is that the same as fibonacci
---
No, prime numbers and Fibonacci numbers are different sequences, though they can sometimes overlap.

The Fibonacci sequence starts with 0 and 1, then each subsequent number is the sum of the two preceding numbers:
0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, ...

The prime number sequence is:
2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, ...

Some numbers appear in both sequences (like 2, 3, 5, and 13), but they follow completely differ

In [ ]:
messages = []

system = "You are a patient math tutor.  Do not directly answer a studen'ts questions.  Guide them to a solution step by step."

add_user_message(messages, "How do I solve 5x+3=2?")

answer = chat(messages, system=system)

answer

"Great! Let's work through this equation step by step. \n\nFirst, can you tell me what our goal is when solving an equation like 5x + 3 = 2? What are we trying to find?\n\nOnce you answer that, I'll guide you through the next step. The key is to think about getting x by itself on one side of the equation."

In [ ]:
messages = []

add_user_message(
    messages,
    "Write a Python function that checks a string to see if it is a palindrome.",
)
answer = chat(
    messages, system="You are a Python engineer who writes very concise code."
)
answer

"```python\ndef is_palindrome(s):\n    return s == s[::-1]\n```\n\nFor case-insensitive palindrome checking:\n```python\ndef is_palindrome(s):\n    s = s.lower()\n    return s == s[::-1]\n```\n\nFor alphanumeric-only palindrome checking:\n```python\ndef is_palindrome(s):\n    s = ''.join(c.lower() for c in s if c.isalnum())\n    return s == s[::-1]\n```"

In [ ]:
messages = []

add_user_message(
    messages,
    "Generate a one sentence movie idea.",
)
answer = chat(messages, temperature=1.0)
answer

"A teenage girl discovers that every time she lies, someone in her small town mysteriously disappears, forcing her to navigate high school social dynamics while desperately trying to bring back the people she's accidentally erased from existence."

In [37]:
messages = []

add_user_message(messages, "Write a one sentence description of a database.")

# stream conversation with Claude
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event)
    
    


RawMessageStartEvent(message=Message(id='msg_01PUz35wu4FZzpLWa4UkW4se', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=4, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A database is an', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' organized collection of structured information or', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' data that', type='text_delta'), index=0, ty

In [41]:
messages = []

add_user_message(messages, "write a 1 sentence description of a fake database")
#proper streaming implementation
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # print(text, end="", flush=True)
        pass
    
    
stream.get_final_message()

ParsedMessage(id='msg_011PdiNK1YjygQyrHo9tn3FM', container=None, content=[ParsedTextBlock(citations=None, text='FakeDB is a lightweight in-memory database that generates realistic mock data for testing applications without requiring actual database connections or persistent storage.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=30, server_tool_use=None, service_tier='standard'))

In [9]:
messages = []

add_user_message(messages, "Is tea or coffee better at breakfast?")
# add_assistant_message(messages, "coffee is better because")
add_assistant_message(messages, "neither is very good")

answer = chat(messages)
answer

"! I think a healthy breakfast should be focused on nutrient-dense foods like fruits and vegetables. Coffee and tea both have caffeine which can be problematic in the morning, though tea generally has less. I'd recommend having a glass of water or fresh juice instead. The most important thing is to fuel your body with wholesome foods to start the day right."

In [13]:
messages = []

add_user_message(messages, "count from 1 to 10")
answer = chat(messages, stop_sequences=[", 6"])
answer  


'1, 2, 3, 4, 5'

In [15]:
messages = []

add_user_message(messages, "generate a simple event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

text


'\n{\n  "Rules": [\n    {\n      "Name": "ec2-state-change-rule",\n      "Description": "Trigger Lambda when EC2 instance state changes",\n      "EventPattern": {\n        "source": ["aws.ec2"],\n        "detail-type": ["EC2 Instance State-change Notification"],\n        "detail": {\n          "state": ["running", "stopped", "terminated"]\n        }\n      },\n      "State": "ENABLED",\n      "Targets": [\n        {\n          "Id": "1",\n          "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessEC2StateChange",\n          "RoleArn": "arn:aws:iam::123456789012:role/EventBridgeRole"\n        }\n      ]\n    }\n  ]\n}\n'

In [16]:
import json

json.loads(text.strip())

{'Rules': [{'Name': 'ec2-state-change-rule',
   'Description': 'Trigger Lambda when EC2 instance state changes',
   'EventPattern': {'source': ['aws.ec2'],
    'detail-type': ['EC2 Instance State-change Notification'],
    'detail': {'state': ['running', 'stopped', 'terminated']}},
   'State': 'ENABLED',
   'Targets': [{'Id': '1',
     'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessEC2StateChange',
     'RoleArn': 'arn:aws:iam::123456789012:role/EventBridgeRole'}]}]}

In [17]:
messages = []

add_user_message(messages, "generate 3 different aws cli commands. each should be very short")
text = chat(messages)
text.strip()


'Here are 3 short AWS CLI commands:\n\n```bash\naws s3 ls\n```\n\n```bash\naws ec2 describe-instances\n```\n\n```bash\naws iam list-users\n```'

In [18]:
from IPython.display import Markdown

Markdown(text)



Here are 3 short AWS CLI commands:

```bash
aws s3 ls
```

```bash
aws ec2 describe-instances
```

```bash
aws iam list-users
```